# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [1]:
%pip install evaluate rouge_score ipywidgets jupyterlab_widgets ipympl


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [1]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

print(DEVICE)

cuda


In [ ]:
from huggingface_hub import login

login("xxxxx")

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [3]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all', cache_dir='/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp-nlp/data')

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [4]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

In [7]:
smoltalk_simplified['train'][1]

{'messages': [{'content': 'Name a type of flower that typically grows in a temperate climate',
   'role': 'user'},
  {'content': 'One type of flower that typically grows in a temperate climate is the rose.',
   'role': 'assistant'}],
 'source': 'openhermes-100k'}

In [8]:
smoltalk_simplified['test'][0]

{'messages': [{'content': 'Correct the verb tense in the following sentence: "I swim every day last week.":\n"I swam every day last week."',
   'role': 'user'},
  {'content': '"I swam every day last week."', 'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

In [9]:
len(smoltalk_simplified['train'])

5000

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [10]:
def format_input_output(example):
  # `messages` is a list of messages, each with a `content` string and a `role`.
  messages = example['messages']
  messages_dict = {
      "system": "",
      "user": "",
      "assistant": "",
  }
  for msg in messages:
    messages_dict[msg['role']] = msg['content']

  # TODO: implement this

  return {"prompt": f"<|im_start|>system\n{messages_dict['system']}<|im_end|>\n<|im_start|>user\n{messages_dict['user']}<|im_end|>\n",
          "response": f"<|im_start|>assistant\n{messages_dict['assistant']}<|im_end|>"}
# """
# <|im_start|>system
# You are a helpful assistant.<|im_end|>
# <|im_start|>user
# Hello!<|im_end|>
# <|im_start|>assistant
# Hi! How can I help you today?<|im_end|>
# <|im_start|>user
# What's the weather?<|im_start|>assistant
# """

Apply the function you implemented to the dataset as a whole.

In [11]:
ds_sft = smoltalk_simplified.map(format_input_output)

Then verify that the dataset now contains the new fields you created.

In [12]:
ds_sft['test'][1]

{'messages': [{'content': 'Add proper hyphenation in the following sentence to improve accuracy:\nThe well known French chef prepared the five course meal.',
   'role': 'user'},
  {'content': 'The well-known French chef prepared the five-course meal.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': '<|im_start|>system\n<|im_end|>\n<|im_start|>user\nAdd proper hyphenation in the following sentence to improve accuracy:\nThe well known French chef prepared the five course meal.<|im_end|>\n',
 'response': '<|im_start|>assistant\nThe well-known French chef prepared the five-course meal.<|im_end|>'}

In [13]:
ds_sft

DatasetDict({
    train: Dataset({
        features: ['messages', 'source', 'prompt', 'response'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source', 'prompt', 'response'],
        num_rows: 400
    })
})

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [15]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    tokenized_prompt = tokenizer.encode(prompt)
    tokenized_response = tokenizer.encode(response)

    # TODO: your work goes here.
    mask = len(tokenized_prompt) * [1] + len(tokenized_response) * [1] # one is atteneded

    return {
        "input_ids": tokenized_prompt + tokenized_response,       # Input token ids of the prompt and response
        "attention_mask": mask,  # Attention mask of the prompt and response
        "labels": [-100] * len(tokenized_prompt) + tokenized_response,          # Output token ids of the prompt (masked) and response
    }

tokenized_ds_sft = DatasetDict()
tokenized_ds_sft["train"] = ds_sft["train"].map(tokenize_helper)
tokenized_ds_sft["test"] = ds_sft["test"].map(tokenize_helper)

print(len(tokenized_ds_sft['train'][0]['input_ids']))
print(len(tokenized_ds_sft['train'][0]['labels']))
print(len(tokenized_ds_sft['train'][0]['attention_mask']))

81
81
81


In [16]:
tokenizer.decode(tokenized_ds_sft['train'][0]['input_ids'])



"<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\nThe chef made more food after the restaurant ran out.<|im_end|>"

As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [17]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x[:max_len])
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = -100
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    batch_input_ids = pad_to_max(input_ids_list, pad_value=tokenizer.pad_token_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

In [18]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    tokenized_ds_sft["train"],
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(dataloader))
print(batch)

{'input_ids': tensor([[    1,  9690,   198,  2683,   359,   354,  5646,   298, 12021, 11173,
            30,  1206,   523,   325,  2711,   351,   253,  1694,   284,   346,
           737,   288, 34013,   357,  2289,   288,   260,  2914,   506,  6388,
            30,     2,   198,     1,  4093,   198,    66,   517,  6262,   451,
          6330,   288,   919,   357,  2807,   288,  1044,    42,   198,   504,
         15238,  7674,   578,   282,  1114,    28,   588,   260, 29012,  1135,
           634,   540,    30,     2,   198,     1,   520,  9531,   198,   504,
         29012,  1135,   540,  1114,   990,   260, 15238,  7674,   578,    30,
             2],
        [    1,  9690,   198,     2,   198,     1,  4093,   198,  5820,   253,
          1502,   282,  8525,   338,  3431,  8759,   281,   253, 22054,  2412,
             2,   198,     1,   520,  9531,   198,  2705,  1502,   282,  8525,
           338,  3431,  8759,   281,   253, 22054,  2412,   314,   260,  8739,
            30,     2

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [19]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [20]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


In [21]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?


## Part 3: Supervised fine-tuning



In [26]:


print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to('cpu')

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=32,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

print(pretrained_trainer)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.5185883045196533,
  "eval_model_preparation_time": 0.0041,
  "eval_rougeL": 0.526031007058275,
  "eval_runtime": 8.4402,
  "eval_samples_per_second": 47.392,
  "eval_steps_per_second": 1.54,
  "epoch": 0
}


In [24]:
dataloader = DataLoader(
    tokenized_ds_sft["test"],
    batch_size=1,
    collate_fn=data_collator
)

test_batch = next(iter(dataloader))

print(test_batch['input_ids'].shape)

output = pretrained_model.generate(input_ids = test_batch['input_ids'].to('cuda:0'), attention_mask = test_batch['attention_mask'].to('cuda:0'), pad_token_id=tokenizer.eos_token_id)
output = tokenizer.decode(output[0, test_batch['input_ids'].shape[1]:])
print(output)

torch.Size([1, 50])

assistant
"I swam every day last week."assistant
assistant
"I


### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [25]:
print(pretrained_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [22]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.cuda.empty_cache()

baseline_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

baseline_trainer = make_trainer(base_model, baseline_training_args)

# TODO: train and evaluate the model.
t0 = time.perf_counter()
baseline_trainer.train()
baseline_eval_metrics = baseline_trainer.evaluate()
baseline_eval_time = time.perf_counter() - t0

baseline_eval_loss = float(baseline_eval_metrics["eval_loss"])
baseline_rougeL = baseline_eval_metrics.get("eval_rougeL", None)

print("\nBASELINE EVAL METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


  0%|          | 0/1570 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

{'eval_loss': '1.26', 'eval_rougeL': '0.6421', 'eval_runtime': '10.05', 'eval_samples_per_second': '39.8', 'eval_steps_per_second': '4.975', 'epoch': '1'}


...

{'eval_loss': '1.132', 'eval_rougeL': '0.6583', 'eval_runtime': '10.02', 'eval_samples_per_second': '39.93', 'eval_steps_per_second': '4.991', 'epoch': '9'}


  0%|          | 0/50 [00:00<?, ?it/s]

{'eval_loss': '1.132', 'eval_rougeL': '0.6574', 'eval_runtime': '9.854', 'eval_samples_per_second': '40.59', 'eval_steps_per_second': '5.074', 'epoch': '10'}
{'train_runtime': '571.4', 'train_samples_per_second': '87.5', 'train_steps_per_second': '2.748', 'train_loss': '1.135', 'epoch': '10'}


  0%|          | 0/50 [00:00<?, ?it/s]


BASELINE EVAL METRICS:
{
  "eval_loss": 1.1322262287139893,
  "eval_rougeL": 0.6574418573019968,
  "eval_runtime": 9.6358,
  "eval_samples_per_second": 41.512,
  "eval_steps_per_second": 5.189,
  "epoch": 10.0
}


In [31]:
dataloader = DataLoader(
    tokenized_ds_sft["test"],
    batch_size=1,
    collate_fn=data_collator
)

test_batch = next(iter(dataloader))

output = base_model.generate(input_ids = test_batch['input_ids'].to('cuda:0'), attention_mask = test_batch['attention_mask'].to('cuda:0'), pad_token_id=tokenizer.eos_token_id)
output = tokenizer.decode(output[0, test_batch['input_ids'].shape[1]:])
print(output)

/mimer/NOBACKUP/groups/oovgen/ziyuan/wasp_env/lib/python3.10/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=70) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


assistant
Correct the verb tense in the following sentence: "I swim every day last week."


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [24]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    num_param = 0
    for param in model.parameters():
        if param.requires_grad:
            param_count = param.numel()
            num_param += param_count
    # TODO: Add your code here
    return num_param

print(f"{num_trainable_parameters(base_model)// 1e6}M")

134.0M


Apply this function to the SFT-trained model and check that the result makes sense.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [27]:
import copy

import torch.nn as nn
import math

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        # TODO: Add your code here
        self.W = W
        self.W.requires_grad_(False)
        output_dim, input_dim = self.W.weight.shape
        self.A = nn.Linear(input_dim, r, bias=False)
        self.B = nn.Linear(r, output_dim, bias=False)
        self.scaling = alpha / r
        nn.init.kaiming_uniform_(
            self.A.weight,
            a=math.sqrt(5)
        )

        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = x @ self.W.weight.T
        lora_output = self.B(self.A(x))
        return original + lora_output * self.scaling
        # TODO: Add your code here
        raise NotImplementedError()


def extract_lora_targets(model):
  # TODO: Add your code here
  # extract Q and V from Transformer Layer
    target_name_list = {}
    for name, _module in model.named_modules():
        if 'self_attn.q_proj' in name or 'self_attn.v_proj' in name:
            target_name_list[name] = LoRALayer(_module, 4, 4)
        
    return target_name_list


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [28]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

new_model = copy.deepcopy(pretrained_model)
named_layers = extract_lora_targets(new_model)
new_model = replace_layers(new_model, named_layers)



### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [29]:
import torch.nn as nn
import math

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        # TODO: Add your code here
        self.W = W
        self.W.requires_grad_(False)
        output_dim, input_dim = self.W.weight.shape
        self.A = nn.Linear(input_dim, r, bias=False)
        self.B = nn.Linear(r, output_dim, bias=False)
        self.scaling = alpha / r
        nn.init.kaiming_uniform_(
            self.A.weight,
            a=math.sqrt(5)
        )

        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = x @ self.W.weight.T
        lora_output = self.B(self.A(x))
        return original + lora_output * self.scaling
        # TODO: Add your code here
        raise NotImplementedError()

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [30]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

lora_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

# base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

lora_trainer = make_trainer(new_model, lora_training_args)

# TODO: train and evaluate the model.
t0 = time.perf_counter()
lora_trainer.train()
lora_eval_metrics = lora_trainer.evaluate()
lora_eval_time = time.perf_counter() - t0

lora_eval_loss = float(lora_eval_metrics["eval_loss"])
lora_rougeL = lora_eval_metrics.get("eval_rougeL", None)

print("\nLoRA EVAL METRICS:")
print(json.dumps(lora_eval_metrics, indent=2))

  0%|          | 0/1570 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

{'eval_loss': '1.284', 'eval_rougeL': '0.6384', 'eval_runtime': '10.24', 'eval_samples_per_second': '39.08', 'eval_steps_per_second': '4.885', 'epoch': '1'}


  0%|          | 0/50 [00:00<?, ?it/s]

{'eval_loss': '1.2', 'eval_rougeL': '0.6459', 'eval_runtime': '10.27', 'eval_samples_per_second': '38.96', 'eval_steps_per_second': '4.87', 'epoch': '2'}


...

{'eval_loss': '1.129', 'eval_rougeL': '0.6545', 'eval_runtime': '10.49', 'eval_samples_per_second': '38.13', 'eval_steps_per_second': '4.767', 'epoch': '9'}


  0%|          | 0/50 [00:00<?, ?it/s]

{'eval_loss': '1.128', 'eval_rougeL': '0.6542', 'eval_runtime': '10.32', 'eval_samples_per_second': '38.76', 'eval_steps_per_second': '4.845', 'epoch': '10'}
{'train_runtime': '630.4', 'train_samples_per_second': '79.31', 'train_steps_per_second': '2.49', 'train_loss': '1.144', 'epoch': '10'}


  0%|          | 0/50 [00:00<?, ?it/s]


LoRA EVAL METRICS:
{
  "eval_loss": 1.1284071207046509,
  "eval_rougeL": 0.6542473632438299,
  "eval_runtime": 10.1361,
  "eval_samples_per_second": 39.463,
  "eval_steps_per_second": 4.933,
  "epoch": 10.0
}


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [32]:
dataloader = DataLoader(
    tokenized_ds_sft["test"],
    batch_size=1,
    collate_fn=data_collator
)

test_batch = next(iter(dataloader))

output = new_model.generate(input_ids = test_batch['input_ids'].to('cuda:0'), attention_mask = test_batch['attention_mask'].to('cuda:0'), pad_token_id=tokenizer.eos_token_id)
output = tokenizer.decode(output[0, test_batch['input_ids'].shape[1]:])
print(output)

assistant
Correct the verb tense in the following sentence: "I swim every day last week."


In [33]:
print(pretrained_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [39]:
def generate_response(model, input_ids, attention_mask):
    """Helper: generate and decode only the response part."""
    device = next(model.parameters()).device
    output = model.generate(
        input_ids=input_ids.to(device),
        attention_mask=attention_mask.to(device),
        max_new_tokens=128,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

# --- Compare on first 5 test samples ---
dataloader = DataLoader(
    tokenized_ds_sft["test"],
    batch_size=1,
    collate_fn=data_collator
)

for idx, test_batch in enumerate(dataloader):
    print("=" * 70)
    print(f" EXAMPLE {idx + 1}")
    print("=" * 70)

    # Prompt
    prompt_text = tokenizer.decode(test_batch['input_ids'][0], skip_special_tokens=True)
    print(f"\n PROMPT:\n{prompt_text[:300]}{'...' if len(prompt_text) > 300 else ''}")

    # Gold reference (from labels, skipping -100)
    ref_ids = test_batch['labels'][0][test_batch['labels'][0] != -100]
    ref_text = tokenizer.decode(ref_ids, skip_special_tokens=True).strip()
    print(f"\n REFERENCE:\n  {ref_text[:200]}")

    # Three models
    print(f"\n PRETRAINED (no training):")
    print(f"  {generate_response(pretrained_model, test_batch['input_ids'], test_batch['attention_mask'])[:200]}")

    print(f"\n FULL SFT (all params trained):")
    print(f"  {generate_response(base_model, test_batch['input_ids'], test_batch['attention_mask'])[:200]}")

    print(f"\n LoRA (Q/V only, r=4):")
    print(f"  {generate_response(new_model, test_batch['input_ids'], test_batch['attention_mask'])[:200]}")

    print()
    if idx >= 10:
        break


 EXAMPLE 1

 PROMPT:
system

user
Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."
assistant
"I swam every day last week."

 REFERENCE:
  assistant
"I swam every day last week."

 PRETRAINED (no training):
  assistant
"I swam every day last week."assistant
assistant
"I swam every day last week."assistant
assistant
"I swam every day last week."assistant
assistant
"I swam every day last week."assistant
assi

 FULL SFT (all params trained):
  assistant
Correct the verb tense in the following sentence: "I swim every day last week.":
"I swim every day last week."assistant
Correct the verb tense in the following sentence: "I swim every day la

 LoRA (Q/V only, r=4):
  assistant
Correct the verb tense in the following sentence: "I swim every day last week.":
"I swim every day last week."assistant
Correct the verb tense in the following sentence: "I swim every day la

 EXAMPLE 2

 PROMPT:
system

user
Add proper hyphenation in the f